# Busca Vetorial (*Vector Search*) em Bancos de Dados (*Databases*) de Embeddings

Este notebook encontra segmentos de áudio que **possuam similaridade com templates de áudios pré-selecionados**, através da comparação dos templates com **embeddings disponíveis em bancos de dados**.

---

### O que este notebook faz (em ordem):
1. **Conecta ao seu Google Drive** para acessar áudios e salvar resultados
2. **Instala os pacotes de software necessários** automaticamente
3. **Extrai embeddings** para cada template de áudio e salva como arquivo `.npy`
4. **Executa uma busca vetorial por similaridade** de cada template em um ou mais **bancos de dados de embeddings** existentes e salva os valores de similaridade em CSV
5. **Extrai um segmento de áudio** para cada resultado, e organiza-os em uma pasta por ponto e rótulo
6. **Revisa os segmentos extraídos** um a um em um espectrograma interativo + player de áudio, marcando cada um como *verdadeiro* ou *falso*

### Modelos predefinidos suportados:
| Modelo | Repositório HuggingFace | Taxa de amostragem | Janela | Tamanho do embedding |
|---|---|---|---|---|
| Perch v2.0 | `biodiversica/Perch-onnx-backbone` | 32 000 Hz | 5.0 s | 1536 |
| BirdNET v2.4 | `biodiversica/BirdNET-onnx-backbone` | 48 000 Hz | 3.0 s | 1024 |

> O backbone que você selecionar aqui **deve corresponder** àquele usado para construir o banco de embeddings em que você fará a busca (mesma taxa de amostragem, duração da janela e tamanho do embedding).

### Templates:
Aponte este notebook para uma pasta do Google Drive com clipes de áudio de exemplo curtos. **O nome do arquivo de cada clipe (sem extensão) se torna seu rótulo.** Para fornecer vários exemplos para um mesmo rótulo, agrupe os clipes em uma subpasta — o nome da subpasta se torna o rótulo e todos os seus clipes são combinados.

### Bancos de dados (*Database*) de embeddings:
Faça a busca em um único arquivo `*.embeddings.db`, ou em uma pasta contendo vários deles, produzidos pelo notebook **Criação de Banco de Dados de Embeddings** (*pt-BR/compute_embeddings_database_pt-BR.ipynb*).

### Resultados:
Para cada banco de dados, um CSV com resultados de similaridade é armazenado na sua pasta de resultados com as colunas
`label, site, date, time, confidence, start_time, end_time` (a coluna `confidence` contém o score de similaridade). As Partes 5 e 6 então extraem e permitem que você revise os segmentos de áudio correspondentes — tudo dentro deste notebook, sem necessidade de outro notebook.

### Como executar:
Execute cada célula **uma de cada vez**, de cima para baixo. Clique em ▶ ou pressione `Shift + Enter`.

> **Primeira vez utilizando notebooks interativos em python?** Uma célula com `[ ]` à esquerda ainda não foi executada. Após executar aparece `[1]`, `[2]`, etc. Se você vir um erro (texto vermelho), leia a mensagem — ela geralmente diz exatamente o que corrigir.

---

Criado por [biodiversica](https://biodiversica.xyz). Para suporte ou feedback, abra uma *Issue* no [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) ou entre em contato através do e-mail **info@biodiversica.xyz**

---
## Parte 1 — Conectar ao Google Drive e Instalar o Software

Execute as duas células abaixo. A primeira pedirá que você **permita o acesso ao seu Google Drive** — clique no link e siga os passos.

In [ ]:
#@title 📂 Conectar ao Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado com sucesso.')

In [ ]:
#@title 📦 Instalar { display-mode: "form" }

!pip install onnxruntime librosa soundfile huggingface_hub pandas matplotlib ipywidgets -q

print('\nTodos os pacotes foram instalados com sucesso.')
print('(O SDK do Arbimon é instalado depois, apenas se você usar o Arbimon como origem do áudio no Parte 5.)')

---
## Parte 2 — Configuração

Preencha os formulários abaixo. **Você não precisa editar nenhum código** — apenas digite ou selecione seus valores em cada formulário e execute a célula.

Execute os três formulários de cima para baixo:
1. **Configurações Gerais** — pasta de templates, caminho dos bancos de dados, pasta de resultados
2. **Modelo** — qual backbone usar (deve corresponder aos bancos de dados em que você fará a busca)
3. **Configurações de Busca** — quantas correspondências manter e como pontuá-las

> **Dica:** Os formulários ocultam o código automaticamente. Para ver o código subjacente, clique no ícone `{ }` no canto superior direito de qualquer célula de formulário.

In [52]:
#@title ⚙️ Configurações Gerais { display-mode: "form" }

import os

#@markdown **Pasta de templates de áudio** — pasta do Google Drive com seus clipes de exemplo.
#@markdown O nome do arquivo de cada clipe se torna seu rótulo; agrupe clipes em uma subpasta para combinar vários
#@markdown exemplos sob um mesmo rótulo (o nome da subpasta). Os embeddings `.npy` calculados são salvos aqui.
DRIVE_TEMPLATES_DIR = '/content/drive/MyDrive/Templates'  #@param {type:"string"}

#@markdown ---
#@markdown **Bancos de dados (*Databases*) de embeddings** — caminho completo para um único arquivo `*.embeddings.db`,
#@markdown **ou** para uma pasta contendo um ou mais deles (busca recursiva).
DRIVE_EMBEDDINGS_DB_PATH = '/content/drive/MyDrive/Embeddings/'  #@param {type:"string"}

#@markdown ---
#@markdown **Pasta para armazenar resultados** — onde os arquivos de resultado da busca serão salvos no seu Google Drive.
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/vector_search_results'  #@param {type:"string"}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print(f'Pasta de templates : {DRIVE_TEMPLATES_DIR}')
if not os.path.isdir(DRIVE_TEMPLATES_DIR):
    print('  AVISO: pasta não encontrada — verifique o caminho e se o Drive está conectado.')
print(f'Caminho dos bancos : {DRIVE_EMBEDDINGS_DB_PATH}')
if not os.path.exists(DRIVE_EMBEDDINGS_DB_PATH):
    print('  AVISO: caminho não encontrado — verifique o caminho e se o Drive está conectado.')
print(f'Pasta de resultados: {DRIVE_RESULTS_DIR}')

In [53]:
#@title 🤖 Modelo { display-mode: "form" }

import os

#@markdown **Predefinição do modelo** — selecione o backbone do modelo.
#@markdown Ele **deve corresponder** ao backbone usado para construir o(s) banco(s) de embeddings em que você fará a busca.
#@markdown - **Perch v2**: backbone Google Perch (32 kHz · janela de 5 s · embeddings de 1536 dimensões)
#@markdown - **BirdNET backbone**: backbone BirdNET v2.4 (48 kHz · janela de 3 s · embeddings de 1024 dimensões)
#@markdown - **Custom ONNX**: forneça seu próprio modelo backbone ONNX
MODEL_PRESET = 'Perch v2'  #@param ["Perch v2", "BirdNET backbone", "Custom ONNX"]

#@markdown ---
#@markdown ### Configurações Custom ONNX *(usadas quando a predefinição = Custom ONNX)*

#@markdown **Origem do modelo personalizado**
CUSTOM_MODEL_SOURCE = 'google_drive'  #@param ["google_drive", "huggingface"]

#@markdown **Caminho no Google Drive** — caminho completo para seu arquivo backbone `.onnx` no Drive.
CUSTOM_DRIVE_MODEL_PATH = '/content/drive/MyDrive/models/my_backbone.onnx'  #@param {type:"string"}

#@markdown **ID do repositório HuggingFace** — ex: `my_org/my-backbone-onnx`
CUSTOM_HF_REPO = ''  #@param {type:"string"}

#@markdown **Nome do arquivo no HuggingFace** — ex: `backbone.onnx`
CUSTOM_HF_FILE = 'backbone.onnx'  #@param {type:"string"}

#@markdown **Taxa de amostragem (Hz)**
CUSTOM_SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Duração da janela (segundos)** — comprimento de cada janela de áudio fornecida ao modelo.
CUSTOM_WINDOW_S = 3.0  #@param {type:"number"}

#@markdown **Nome do *node* de entrada do modelo**
CUSTOM_INPUT_NAME = 'input'  #@param {type:"string"}

#@markdown **Nome do *node* de saída do modelo**
CUSTOM_OUTPUT_NAME = 'embedding'  #@param {type:"string"}

#@markdown **Tamanho do embedding** — número de dimensões no vetor de saída.
CUSTOM_EMBEDDING_SIZE = 1024  #@param {type:"integer"}

_PRESETS = {
    'Perch v2': {
        'hf_repo':        'biodiversica/Perch-onnx-backbone',
        'hf_file':        'perch_v2_backbone.onnx',
        'sample_rate':    32000,
        'window_s':       5.0,
        'input_name':     'inputs',
        'output_name':    'embedding',
        'embedding_size': 1536,
        'model_name':     'perch_v2_backbone',
    },
    'BirdNET backbone': {
        'hf_repo':        'biodiversica/BirdNET-onnx-backbone',
        'hf_file':        'model_backbone.onnx',
        'sample_rate':    48000,
        'window_s':       3.0,
        'input_name':     'INPUT',
        'output_name':    'embedding',
        'embedding_size': 1024,
        'model_name':     'model_backbone',
    },
}

if MODEL_PRESET in _PRESETS:
    _p = _PRESETS[MODEL_PRESET]
    HF_REPO        = _p['hf_repo']
    HF_FILE        = _p['hf_file']
    SAMPLE_RATE    = _p['sample_rate']
    WINDOW_S       = _p['window_s']
    INPUT_NAME     = _p['input_name']
    OUTPUT_NAME    = _p['output_name']
    EMBEDDING_SIZE = _p['embedding_size']
    MODEL_NAME     = _p['model_name']
else:
    SAMPLE_RATE    = CUSTOM_SAMPLE_RATE
    WINDOW_S       = CUSTOM_WINDOW_S
    INPUT_NAME     = CUSTOM_INPUT_NAME
    OUTPUT_NAME    = CUSTOM_OUTPUT_NAME
    EMBEDDING_SIZE = CUSTOM_EMBEDDING_SIZE
    if CUSTOM_MODEL_SOURCE == 'huggingface':
        HF_REPO    = CUSTOM_HF_REPO.strip()
        HF_FILE    = CUSTOM_HF_FILE.strip()
        MODEL_NAME = os.path.splitext(HF_FILE)[0]
    else:
        HF_REPO    = None
        HF_FILE    = None
        MODEL_NAME = os.path.splitext(os.path.basename(CUSTOM_DRIVE_MODEL_PATH))[0]

WINDOW_SAMPLES = int(WINDOW_S * SAMPLE_RATE)

print(f'Predefinição    : {MODEL_PRESET}')
print(f'Nome do modelo  : {MODEL_NAME}')
print(f'Taxa amostragem : {SAMPLE_RATE} Hz')
print(f'Duração janela  : {WINDOW_S} s  ({WINDOW_SAMPLES} amostras)')
print(f'Tam. embedding  : {EMBEDDING_SIZE}')
if HF_REPO:
    print(f'Repo HF         : {HF_REPO}')
    print(f'Arquivo HF      : {HF_FILE}')

In [54]:
#@title 🔎 Configurações de Busca { display-mode: "form" }

#@markdown **Top-K** — quantas das janelas mais semelhantes serão consideradas.
#@markdown No modo *per_label* é por rótulo; no modo *global* é entre todos os rótulos combinados.
TOP_K = 10  #@param {type:"integer"}

#@markdown **Métrica** — como a similaridade entre os embeddings é medida (todas orientadas para maior = mais semelhante).
METRIC = 'cosine'  #@param ["cosine", "dot", "euclidean"]

#@markdown **Modo**
#@markdown - **per_label** — mantém as Top-K janelas para *cada* rótulo independentemente
#@markdown - **global** — mantém as Top-K janelas entre *todos* os rótulos combinados
MODE = 'per_label'  #@param ["per_label", "global"]

#@markdown **Limiar do score de similaridade** — score mínimo de similaridade para manter uma janela (0.0 mantém todas as Top-K).
SCORE_THRESHOLD = 0.7  #@param {type:"number"}

#@markdown **Tamanho do lote** — número de janelas do banco analisadas de uma vez (controla o uso de memória).
BATCH_SIZE = 10000  #@param {type:"integer"}

#@markdown ---
#@markdown **Sobreposição da janela do template (0.0–0.9)** — quando um template de áudio é mais longo que a janela
#@markdown do modelo, define a sobreposição entre as janelas extraídas do template. Mais sobreposição = mais
#@markdown embeddings por rótulo.
TEMPLATE_OVERLAP = 0.0  #@param {type:"slider", min:0.0, max:0.9, step:0.1}

print(f'Top-K            : {TOP_K}')
print(f'Métrica          : {METRIC}')
print(f'Modo             : {MODE}')
print(f'Limiar de similaridade : {SCORE_THRESHOLD}')
print(f'Tamanho do lote  : {BATCH_SIZE}')
print(f'Sobrep. template : {TEMPLATE_OVERLAP}')

---
## Parte 3 — Extrair embeddings dos templates

Esta célula carrega o modelo backbone e extrai embeddings para cada clipe de áudio de exemplo na sua
pasta de templates. Os embeddings é salvo como um arquivo `.npy`.

- Se um template for mais longo que uma janela do modelo, diferentes embeddings são extraidas e armazenadas — cada uma se torna um
  exemplo independente para aquele rótulo (durante a busca, seleciona-se o exemplo com melhor similaridade).
- Se um arquivo `.npy` já existir para um template, esta parte de extração é **ignorada**, e o arquivo existente é utilizado nos próximos passos deste notebook.

**Rótulos:** um áudio colocado diretamente na pasta de templates é rotulado pelo nome do arquivo. Um áudio dentro
de uma subpasta é rotulado pelo **nome da subpasta**, então vários clipes na mesma subpasta compartilham um rótulo.

In [55]:
#@title 🧠 Carregar modelo e extrair embeddings dos templates { display-mode: "form" }

import os
import glob as _glob
import numpy as np
import librosa

# --- Resolver e carregar o modelo ONNX ---
if HF_REPO:
    from huggingface_hub import hf_hub_download
    print(f'Baixando backbone do HuggingFace: {HF_REPO} / {HF_FILE} ...')
    _model_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
else:
    _model_path = CUSTOM_DRIVE_MODEL_PATH

if not os.path.exists(_model_path):
    raise FileNotFoundError(f'Modelo não encontrado: {_model_path}\nVerifique a configuração do modelo no Parte 2.')

import onnxruntime as ort
_ort_session = ort.InferenceSession(_model_path)
_out_names   = [o.name for o in _ort_session.get_outputs()]
_fetch_output = OUTPUT_NAME if OUTPUT_NAME in _out_names else _out_names[0]
if _fetch_output != OUTPUT_NAME:
    print(f"AVISO: saída '{OUTPUT_NAME}' não encontrada no modelo; usando '{_fetch_output}' no lugar.")

print('Modelo ONNX carregado.')
print(f'  Entradas : {[i.name for i in _ort_session.get_inputs()]}  →  usando \'{INPUT_NAME}\'')
print(f'  Saídas   : {_out_names}  →  obtendo \'{_fetch_output}\'')
print(f'  Provedores: {_ort_session.get_providers()}')

_template_stride = max(1, int(WINDOW_SAMPLES * (1 - TEMPLATE_OVERLAP)))

def _embed_window(audio_window):
    seg = audio_window.astype(np.float32)
    if len(seg) < WINDOW_SAMPLES:
        seg = np.pad(seg, (0, WINDOW_SAMPLES - len(seg)))
    out = _ort_session.run([_fetch_output], {INPUT_NAME: seg[np.newaxis, :]})[0]
    return out.squeeze(0).astype(np.float32)

def _embed_clip(audio):
    vecs = []
    for start in range(0, max(1, len(audio)), _template_stride):
        seg = audio[start:start + WINDOW_SAMPLES]
        if len(seg) < WINDOW_SAMPLES * 0.5 and vecs:
            continue
        vecs.append(_embed_window(seg))
    return np.stack(vecs) if vecs else None

def _label_for(audio_path, templates_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(templates_dir)
    if parent != root:
        # clipe está em uma subpasta → o rótulo é o nome da subpasta
        return os.path.basename(parent)
    return os.path.splitext(os.path.basename(audio_path))[0]

if not os.path.isdir(DRIVE_TEMPLATES_DIR):
    raise FileNotFoundError(f'Pasta de templates não encontrada: {DRIVE_TEMPLATES_DIR}')

_clips = sorted(
    _glob.glob(os.path.join(DRIVE_TEMPLATES_DIR, '**', '*.wav'),  recursive=True) +
    _glob.glob(os.path.join(DRIVE_TEMPLATES_DIR, '**', '*.flac'), recursive=True) +
    _glob.glob(os.path.join(DRIVE_TEMPLATES_DIR, '**', '*.mp3'),  recursive=True)
)

if not _clips:
    raise FileNotFoundError(f'Nenhum template de áudio (.wav/.flac/.mp3) encontrado em: {DRIVE_TEMPLATES_DIR}')

print(f'\nEncontrado(s) {len(_clips)} template(s) de áudio. Dimensão do embedding: {EMBEDDING_SIZE}')
print('=' * 65)

_computed = 0
_skipped  = 0
for _clip in _clips:
    _npy = os.path.splitext(_clip)[0] + '.npy'
    _label = _label_for(_clip, DRIVE_TEMPLATES_DIR)
    if os.path.exists(_npy):
        print(f'  [ignorado] {os.path.relpath(_clip, DRIVE_TEMPLATES_DIR)}  (rótulo: {_label})')
        _skipped += 1
        continue
    try:
        _audio, _ = librosa.load(_clip, sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'  ERRO ao carregar {os.path.basename(_clip)}: {e} — ignorando.')
        continue
    _emb = _embed_clip(_audio)
    if _emb is None:
        print(f'  AVISO: {os.path.basename(_clip)} curto demais para extrair embeddings — ignorando.')
        continue
    np.save(_npy, _emb)
    print(f'  [feito] {os.path.relpath(_clip, DRIVE_TEMPLATES_DIR)}  (rótulo: {_label}, {_emb.shape[0]} janela(s))')
    _computed += 1

print('=' * 65)
print(f'Embeddings dos templates — extraídos: {_computed}, ignorados (existentes): {_skipped}')
print('Continue para a Parte 4 para executar a busca.')

---
## Parte 4 — Executar a busca vetorial (*vector search*)

Esta célula carrega cada template `.npy` e, para cada banco de embeddings, calcula a similaridade entre os templates e cada janela armazenada, mantendo as melhores correspondências (Top-K) de acordo com suas configurações de busca.

Para cada banco de dados, um CSV de resultados é armazenado na sua pasta de resultados com o nome `<banco>.search.csv` com as colunas
`label, site, date, time, confidence, start_time, end_time` (mais `recording_id` como referência).

A coluna `confidence` contém o score de similaridade. Continue para a **Parte 5** para extrair os segmentos de áudio
correspondentes e a **Parte 6** para revisá-los diretamente neste notebook.

In [56]:
#@title 🚀 Executar { display-mode: "form" }

import os
import csv
import glob as _glob
import heapq
import sqlite3
from datetime import datetime
from pathlib import Path
import numpy as np

# ── Carregar templates dos arquivos .npy salvos no Parte 3 ───────────────────────
_template_npys = sorted(_glob.glob(os.path.join(DRIVE_TEMPLATES_DIR, '**', '*.npy'), recursive=True))
if not _template_npys:
    raise FileNotFoundError('Nenhum arquivo de template .npy encontrado. Execute o Parte 3 primeiro.')

templates = {}  # rótulo -> (n_exemplos, dim) float32
for _npy in _template_npys:
    _arr = np.load(_npy).astype(np.float32)
    if _arr.ndim == 1:
        _arr = _arr[np.newaxis, :]
    _label = _label_for(_npy, DRIVE_TEMPLATES_DIR)
    templates.setdefault(_label, []).append(_arr)
templates = {lbl: np.vstack(parts) for lbl, parts in templates.items()}

_label_list = list(templates.keys())
_template_dim = next(iter(templates.values())).shape[1]
_dims = {a.shape[1] for a in templates.values()}
if len(_dims) > 1:
    raise ValueError(f'Os templates têm dimensões inconsistentes: {_dims}')

# Matriz plana de templates + lista de rótulos paralela, e índices de coluna por rótulo.
_flat_templates = np.vstack([templates[l] for l in _label_list])
_template_labels = [l for l in _label_list for _ in range(templates[l].shape[0])]
_label_cols = {}
for _ci, _lbl in enumerate(_template_labels):
    _label_cols.setdefault(_lbl, []).append(_ci)
_label_cols = {l: np.array(idxs) for l, idxs in _label_cols.items()}

print(f'{sum(t.shape[0] for t in templates.values())} vetor(es) de template carregado(s) '
      f'entre {len(_label_list)} rótulo(s): {_label_list}')
print(f'Dimensão do template: {_template_dim}')

# ── Funções de pontuação (cosine / dot / euclidean) ───────────────────────────────
def _compute_scores(batch, tmpl, metric):
    if metric == 'cosine':
        b = batch / (np.linalg.norm(batch, axis=1, keepdims=True) + 1e-8)
        t = tmpl / (np.linalg.norm(tmpl, axis=1, keepdims=True) + 1e-8)
        return (b @ t.T).astype(np.float32)
    if metric == 'dot':
        return (batch @ tmpl.T).astype(np.float32)
    if metric == 'euclidean':
        b_sq = np.sum(batch ** 2, axis=1, keepdims=True)
        t_sq = np.sum(tmpl ** 2, axis=1)
        dist_sq = b_sq + t_sq - 2.0 * (batch @ tmpl.T)
        return (-np.sqrt(np.maximum(dist_sq, 0.0))).astype(np.float32)
    raise ValueError(f"Métrica desconhecida '{metric}'.")

def _iter_embeddings_batched(db_path, batch_size):
    conn = sqlite3.connect(db_path)
    try:
        cur = conn.cursor()
        cur.execute('SELECT site_name, recording_id, chunk_index, datetime, data '
                    'FROM embeddings ORDER BY site_name, recording_id, chunk_index')
        while True:
            rows = cur.fetchmany(batch_size)
            if not rows:
                break
            keys = [(r[0], r[1], r[2], r[3]) for r in rows]
            matrix = np.stack([np.frombuffer(r[4], dtype=np.float32).copy() for r in rows])
            yield keys, matrix
    finally:
        conn.close()

def _load_db_metadata(db_path):
    conn = sqlite3.connect(db_path)
    try:
        return dict(conn.execute('SELECT key, value FROM metadata').fetchall())
    finally:
        conn.close()

def _chunk_times(chunk_index, window_s, sample_rate, overlap):
    stride = max(1, int(window_s * sample_rate * (1.0 - overlap)))
    start = chunk_index * stride / sample_rate
    return start, start + window_s

_RESULT_COLS = ['label', 'site', 'date', 'time', 'confidence',
                'start_time', 'end_time', 'recording_id']

def _row_from_key(key, label, score, window_s, sample_rate, overlap):
    site_name, recording_id, chunk_index, dt = key
    start_t, end_t = _chunk_times(chunk_index, window_s, sample_rate, overlap)
    date_str, time_str = '', ''
    iso = dt or ''
    if not iso:
        # tenta extrair um timestamp do id da gravação (ex: SITE_20250628_185902)
        import re as _re
        m = _re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', recording_id) or \
            _re.search(r'(\d{8})_(\d{6})', recording_id)
        if m:
            g = m.groups()
            if len(g) == 6:
                iso = f'{g[0]}-{g[1]}-{g[2]}T{g[3]}:{g[4]}:{g[5]}'
            else:
                d, t = g
                iso = f'{d[:4]}-{d[4:6]}-{d[6:8]}T{t[:2]}:{t[2:4]}:{t[4:6]}'
    if iso:
        try:
            _d = datetime.fromisoformat(iso)
            date_str = _d.strftime('%Y-%m-%d')
            time_str = _d.strftime('%H:%M:%S')
        except Exception:
            pass
    return {
        'label':      label,
        'site':       site_name or '',
        'date':       date_str,
        'time':       time_str,
        'confidence': round(float(score), 8),
        'start_time': round(start_t, 6),
        'end_time':   round(end_t, 6),
        'recording_id': recording_id,
    }

def _search_one_db(db_path):
    meta = _load_db_metadata(db_path)
    emb_size = int(meta.get('embedding_size', 0))
    if emb_size and emb_size != _template_dim:
        print(f'  IGNORADO — embedding_size {emb_size} do BD != dim {_template_dim} do template '
              '(backbone incorreto para este banco de dados).')
        return None
    sample_rate = int(float(meta.get('sample_rate', SAMPLE_RATE)))
    window_s    = float(meta.get('window_duration_s', WINDOW_S))
    overlap     = float(meta.get('overlap', 0.0))

    if MODE == 'per_label':
        heaps = {lbl: [] for lbl in _label_list}
    else:
        gheap = []
    counter = 0

    for keys, matrix in _iter_embeddings_batched(db_path, BATCH_SIZE):
        scores = _compute_scores(matrix, _flat_templates, METRIC)  # (B, T)
        B = len(keys)
        if MODE == 'per_label':
            for lbl, cols in _label_cols.items():
                lbl_scores = np.max(scores[:, cols], axis=1)
                heap = heaps[lbl]
                for bi in np.where(lbl_scores >= SCORE_THRESHOLD)[0]:
                    sc = float(lbl_scores[bi]); counter += 1
                    if len(heap) < TOP_K:
                        heapq.heappush(heap, (sc, counter, keys[int(bi)]))
                    elif sc > heap[0][0]:
                        heapq.heapreplace(heap, (sc, counter, keys[int(bi)]))
        else:
            all_scores = np.stack([np.max(scores[:, _label_cols[l]], axis=1) for l in _label_list])
            best_idx = np.argmax(all_scores, axis=0)
            best_sc  = all_scores[best_idx, np.arange(B)]
            for bi in np.where(best_sc >= SCORE_THRESHOLD)[0]:
                sc = float(best_sc[bi]); lbl = _label_list[int(best_idx[bi])]; counter += 1
                if len(gheap) < TOP_K:
                    heapq.heappush(gheap, (sc, counter, keys[int(bi)], lbl))
                elif sc > gheap[0][0]:
                    heapq.heapreplace(gheap, (sc, counter, keys[int(bi)], lbl))

    rows = []
    if MODE == 'per_label':
        for lbl, heap in heaps.items():
            for sc, _, key in sorted(heap, key=lambda x: -x[0]):
                rows.append(_row_from_key(key, lbl, sc, window_s, sample_rate, overlap))
        rows.sort(key=lambda r: (r['label'], -r['confidence']))
    else:
        for sc, _, key, lbl in sorted(gheap, key=lambda x: -x[0]):
            rows.append(_row_from_key(key, lbl, sc, window_s, sample_rate, overlap))
    return rows

# ── Descobrir bancos de dados ───────────────────────────────────────────────────────
if os.path.isdir(DRIVE_EMBEDDINGS_DB_PATH):
    _dbs = sorted(_glob.glob(os.path.join(DRIVE_EMBEDDINGS_DB_PATH, '**', '*.db'), recursive=True))
elif os.path.isfile(DRIVE_EMBEDDINGS_DB_PATH):
    _dbs = [DRIVE_EMBEDDINGS_DB_PATH]
else:
    raise FileNotFoundError(f'Caminho dos bancos de dados não encontrado: {DRIVE_EMBEDDINGS_DB_PATH}')

if not _dbs:
    raise FileNotFoundError(f'Nenhum arquivo .db encontrado em: {DRIVE_EMBEDDINGS_DB_PATH}')

print(f'\nBancos de dados para buscar: {len(_dbs)}')
print('=' * 65)

_total_rows = 0
for _db in _dbs:
    _name = os.path.basename(_db)
    print(f'\n{_name}')
    _rows = _search_one_db(_db)
    if _rows is None:
        continue
    _stem = os.path.basename(_db)
    for _suf in ('.embeddings.db', '.db'):
        if _stem.endswith(_suf):
            _stem = _stem[:-len(_suf)]
            break
    _out = os.path.join(DRIVE_RESULTS_DIR, f'{_stem}.search.csv')
    with open(_out, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=_RESULT_COLS)
        w.writeheader()
        w.writerows(_rows)
    _total_rows += len(_rows)
    print(f'  → {len(_rows)} correspondência(s) armazenadas(s) em {_out}')

print('\n' + '=' * 65)
print(f'Busca vetorial concluída — {_total_rows} correspondência(s) no total em {len(_dbs)} banco(s) de dados.')
print(f'Pasta de resultados: {DRIVE_RESULTS_DIR}')
print('\nContinue para a Parte 5 para extrair os segmentos correspondentes, depois a Parte 6 para revisá-los.')

---
## Parte 5 — Extrair segmentos de áudio

Esta parte extrai segmentos de áudio para cada correspondência encontrada na Parte 4 e salva no seu Google Drive, em uma subpasta por ponto e rótulo.

**Como funciona:**
- Cada correspondência armazena o ponto de gravação, a data, o horário local e a posição exata (em segundos) dentro da gravação.
- O notebook localiza a gravação original na sua pasta do Google Drive (ou baixa de um projeto do Arbimon),
  extrai a janela de áudio correspondente e salva como um arquivo `.wav`.

**Antes de executar:** complete a Parte 4 para que os arquivos de resultado existam. Depois execute as células abaixo de cima para baixo.

> **Dica:** use as configurações de *espécie* e *máximo número de segmentos por rótulo* abaixo para limitar quantos segmentos são extraídos.

In [57]:
#@title 📋 Carregar resultados da busca { display-mode: "form" }
#@markdown Lê os arquivos de resultado gravados na Parte 4 e prepara para extração.
#@markdown
#@markdown **Score mínimo** — descarta correspondências abaixo deste score de similaridade antes de extrair (0.0 mantém todas).
MIN_SCORE = 0.7  #@param {type:"number"}

import os, glob, sqlite3
import pandas as pd

_csvs = sorted(glob.glob(os.path.join(DRIVE_RESULTS_DIR, '**', '*.search.csv'), recursive=True))
if not _csvs:
    raise FileNotFoundError(
        f'Nenhum arquivo "*.search.csv" encontrado em {DRIVE_RESULTS_DIR}. Execute o Parte 4 primeiro.')

_frames = [pd.read_csv(c) for c in _csvs]
df_raw = pd.concat(_frames, ignore_index=True)

# Mapeia as colunas da busca para os nomes padrão internos usados pelos Partes 5/6.
df = df_raw.rename(columns={
    'label':      'Common name',
    'site':       'Point name',
    'date':       'Date',
    'time':       'Time',
    'confidence': 'Confidence',
})

df['Date']       = pd.to_datetime(df['Date'], errors='coerce').dt.date
df['Time']       = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce').dt.time
df['Confidence'] = pd.to_numeric(df['Confidence'], errors='coerce')
df['start_time'] = pd.to_numeric(df['start_time'], errors='coerce')
df['end_time']   = pd.to_numeric(df['end_time'], errors='coerce')

if MIN_SCORE > 0:
    df = df[df['Confidence'] >= MIN_SCORE]

# Enriquece com stream_id / utc_offset das tabelas `sites` dos bancos de dados para que
# a extração do Arbimon possa converter o horário local de volta para UTC. Inofensivo para o Drive.
_site_info = {}  # site_name -> (stream_id, utc_offset_int)
if os.path.isdir(DRIVE_EMBEDDINGS_DB_PATH):
    _db_files = glob.glob(os.path.join(DRIVE_EMBEDDINGS_DB_PATH, '**', '*.db'), recursive=True)
elif os.path.isfile(DRIVE_EMBEDDINGS_DB_PATH):
    _db_files = [DRIVE_EMBEDDINGS_DB_PATH]
else:
    _db_files = []
for _db in _db_files:
    try:
        _con = sqlite3.connect(_db)
        for _sn, _sid, _off in _con.execute('SELECT site_name, stream_id, utc_offset FROM sites'):
            _site_info[_sn] = (_sid or '', int(round(_off or 0)))
        _con.close()
    except Exception:
        pass

df['stream_id']  = df['Point name'].map(lambda s: _site_info.get(s, ('', 0))[0])
df['utc_offset'] = df['Point name'].map(lambda s: f"UTC{_site_info.get(s, ('', 0))[1]:+d}")

print(f'Arquivos de resultado carregados : {len(_csvs)}')
print(f'Total de correspondências        : {len(df)}')
if not df.empty:
    print(f'Rótulos                          : {sorted(df["Common name"].dropna().unique().tolist())}')
    print(f'Pontos                           : {sorted(df["Point name"].dropna().unique().tolist())}')
    print(f'Faixa de escore                  : {df["Confidence"].min():.4f} -> {df["Confidence"].max():.4f}')
else:
    print('Nenhuma correspondência restante após o filtro de escore mínimo.')

In [58]:
#@title ⚙️ Configurações de extração { display-mode: "form" }

#@markdown **Origem do áudio** — onde estão armazenadas as gravações originais?
AUDIO_SOURCE = "google_drive"  #@param ["google_drive", "arbimon"]

#@markdown ---
#@markdown ### Se a origem do áudio = Google Drive
#@markdown Caminho completo para a pasta contendo os arquivos de áudio no seu Drive.
#@markdown Subpastas com os nomes dos pontos são pesquisadas automaticamente.
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/audio/"  #@param {type:"string"}

#@markdown ---
#@markdown **Pasta de saída** — onde os segmentos extraídos serão salvos no seu Drive.
#@markdown Os segmentos são organizados em uma subpasta por ponto, depois por rótulo.
DRIVE_SEGMENTS_DIR = "/content/drive/MyDrive/vector_search_segments"  #@param {type:"string"}

#@markdown ---
#@markdown **Rótulos a extrair** — deixe em branco para extrair todos os rótulos dos resultados.
#@markdown Separe os nomes com vírgulas.  Exemplo: `BOAALB, PHYLUT`
EXTRACT_SPECIES = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Máximo número de segmentos por rótulo** — limita quantos segmentos são extraídos para cada rótulo.
#@markdown Defina como **0** para extrair todas as correspondências (sem limite).
MAX_SEGMENTS_PER_LABEL = 10  #@param {type:"integer"}

#@markdown **Semente aleatória** — controla quais correspondências são selecionadas quando o limite acima é aplicado.
#@markdown Use qualquer inteiro positivo para resultados reproduzíveis, ou **-1** para um sorteio diferente a cada vez.
RANDOM_SEED = -1  #@param {type:"integer"}

#@markdown ---
#@markdown **Preenchimento (segundos)** — áudio extra adicionado antes e depois de cada janela correspondente.
SEGMENT_PADDING_S = 0.0  #@param {type:"number"}

#@markdown **Taxa de amostragem (Hz)** — taxa para os arquivos WAV salvos.
#@markdown Use a mesma taxa do seu modelo (ex: 48000 para BirdNET 2.4, 32000 para Perch 2.0).
EXTRACT_SR = 48000  #@param {type:"integer"}

import os
os.makedirs(DRIVE_SEGMENTS_DIR, exist_ok=True)
print(f'Origem do áudio        : {AUDIO_SOURCE}')
print(f'Pasta de saída         : {DRIVE_SEGMENTS_DIR}')
print(f'Máx. segmentos / rótulo: {"todos" if MAX_SEGMENTS_PER_LABEL == 0 else MAX_SEGMENTS_PER_LABEL}')
print(f'Semente aleatória      : {"off (aleatória)" if RANDOM_SEED < 0 else RANDOM_SEED}')
print(f'Preenchimento          : {SEGMENT_PADDING_S} s')
print(f'Taxa de amostragem     : {EXTRACT_SR} Hz')
if EXTRACT_SPECIES.strip():
    print(f'Filtro de rótulo       : {[x.strip() for x in EXTRACT_SPECIES.split(",") if x.strip()]}')
else:
    print('Filtro de rótulo       : todos (dos resultados carregados)')

In [59]:
#@title 🔑 Entrar no Arbimon (pule se estiver usando o Google Drive) { display-mode: "form" }
#@markdown Execute esta célula apenas se a sua origem de áudio for **Arbimon**.
CREDENTIALS_PATH = '/content/drive/MyDrive/rfcx/.rfcx_credentials'  #@param {type:"string"}

import os
if AUDIO_SOURCE == 'arbimon':
    # Instala o SDK do Arbimon aqui (não no topo) para que ele só seja baixado
    # quando você realmente usar o Arbimon como origem do áudio.
    !wget -q https://github.com/rfcx/rfcx-sdk-python/releases/download/0.3.1/rfcx-0.3.1-py3-none-any.whl -O /tmp/rfcx-0.3.1-py3-none-any.whl
    !pip install /tmp/rfcx-0.3.1-py3-none-any.whl -q
    import rfcx
    client_extract = rfcx.Client()
    if os.path.exists(CREDENTIALS_PATH):
        print('Arquivo de credenciais encontrado — entrando automaticamente...')
        client_extract.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    else:
        print('Nenhuma credencial salva encontrada.')
        print('Uma URL aparecerá abaixo — abra-a no seu navegador e faça login com sua conta Arbimon.')
        print('-' * 60)
        client_extract.authenticate(persisted_credentials_path=CREDENTIALS_PATH)
    print('\nConectado ao Arbimon com sucesso.')
else:
    print('A origem do áudio é o Google Drive — login no Arbimon não é necessário. Você pode pular esta célula.')

In [60]:
#@title ⬇️ Acessar gravações { display-mode: "form" }


import re, glob, datetime, unicodedata
import pandas as pd

# Construir subconjunto para extração
df_ext = df.copy()
if EXTRACT_SPECIES.strip():
    _ext_sp = [x.strip() for x in EXTRACT_SPECIES.split(',') if x.strip()]
    df_ext  = df_ext[df_ext['Common name'].isin(_ext_sp)]

if df_ext.empty:
    raise ValueError(
        'Nenhuma correspondência atende aos critérios de extração.\n'
        'Verifique sua lista de rótulos e o escore mínimo nas células acima.'
    )

# ── Amostragem aleatória por rótulo ─────────────────────────────────────────────────
if MAX_SEGMENTS_PER_LABEL > 0:
    _seed = RANDOM_SEED if RANDOM_SEED >= 0 else None
    df_ext = (
        df_ext
        .groupby('Common name', group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), MAX_SEGMENTS_PER_LABEL), random_state=_seed))
    )
    df_ext = df_ext.sort_index()
    print(f'Amostragem aplicada   : até {MAX_SEGMENTS_PER_LABEL} por rótulo'
          f'  (semente={_seed if _seed is not None else "aleatória"})')
    for sp, grp in df_ext.groupby('Common name'):
        print(f'  {sp}: {len(grp)} correspondência(s) selecionada(s)')
else:
    print('Amostragem             : desativada (extraindo todas as correspondências)')

print(f'\nCorrespondências a extrair : {len(df_ext)}')

def _utc_hours(utc_str):
    try:
        return int(str(utc_str).replace('UTC', '').strip() or 0)
    except ValueError:
        return 0

def _rec_datetime(row):
    # Retorna o datetime do início da gravação que contém esta correspondência.
    # Usado apenas para localizar o arquivo de áudio — start_time/end_time são
    # deslocamentos a partir do início desse arquivo (0 = início da gravação).
    # Para Arbimon: converte o horário local para UTC usando utc_offset.
    # Para Google Drive: mantém o horário local (os nomes de arquivo usam horário local).
    try:
        local_dt = datetime.datetime.combine(row['Date'], row['Time'])
        if AUDIO_SOURCE == 'arbimon':
            local_dt = local_dt - datetime.timedelta(hours=_utc_hours(row.get('utc_offset', 'UTC+0')))
        return local_dt
    except Exception:
        return None

df_ext = df_ext.copy()
df_ext['_rec_dt'] = df_ext.apply(_rec_datetime, axis=1)

if AUDIO_SOURCE == 'arbimon':
    unique_recs = df_ext[['stream_id', '_rec_dt']].dropna().drop_duplicates()
else:
    # Mantém o ponto: dois pontos frequentemente gravam no mesmo horário agendado,
    # então a gravação só é identificada de forma única por (ponto, datetime).
    unique_recs = df_ext[['Point name', '_rec_dt']].dropna().drop_duplicates()

print(f'Gravações únicas necessárias : {len(unique_recs)}')

AUDIO_TEMP_DIR = '/content/audio_extract'
os.makedirs(AUDIO_TEMP_DIR, exist_ok=True)

audio_file_map = {}  # chave → caminho do arquivo de áudio local

# ── Arbimon: uma chamada de download por gravação necessária ──────────────────────────
if AUDIO_SOURCE == 'arbimon':
    dest = AUDIO_TEMP_DIR
    os.makedirs(dest, exist_ok=True)

    def _arbimon_file_dt(filepath):
        name = os.path.basename(filepath)
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            return datetime.datetime(*[int(g) for g in m.groups()])
        return None

    for _, rec in unique_recs.iterrows():
        sid = str(rec['stream_id'])
        t0  = rec['_rec_dt']
        key = (sid, t0)

        existing = (glob.glob(f'{dest}/**/*.wav',  recursive=True) +
                    glob.glob(f'{dest}/**/*.flac', recursive=True))
        already = next(
            (f for f in existing
             if sid in os.path.basename(f)
             and (fdt := _arbimon_file_dt(f)) is not None
             and abs((fdt - t0).total_seconds()) < 2),
            None
        )
        if already:
            audio_file_map[key] = already
            print(f'  Em cache : {os.path.basename(already)}')
            continue

        q_min = t0 - datetime.timedelta(seconds=5)
        q_max = t0 + datetime.timedelta(seconds=55)
        try:
            client_extract.download_segments(
                dest_path=dest, stream=sid,
                min_date=q_min, max_date=q_max, parallel=False,
            )
        except Exception as e:
            print(f'  ERRO — stream {sid} em {t0}: {e}')
            continue

        all_downloaded = (glob.glob(f'{dest}/**/*.wav',  recursive=True) +
                          glob.glob(f'{dest}/**/*.flac', recursive=True))
        matched = next(
            (f for f in all_downloaded
             if sid in os.path.basename(f)
             and (fdt := _arbimon_file_dt(f)) is not None
             and abs((fdt - t0).total_seconds()) < 2),
            None
        )
        if matched:
            audio_file_map[key] = matched
            print(f'  Baixado : {os.path.basename(matched)}')
        else:
            print(f'  AVISO: nenhum arquivo encontrado para o stream {sid} em {t0}')

# ── Google Drive: corresponder arquivos por datetime no nome, dentro da pasta do ponto ─
elif AUDIO_SOURCE == 'google_drive':
    all_audio = (
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.wav',  recursive=True) +
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.flac', recursive=True) +
        glob.glob(f'{DRIVE_AUDIO_DIR}/**/*.mp3',  recursive=True)
    )
    print(f'Arquivos de áudio no Drive : {len(all_audio)}')

    def _file_dt(filepath):
        name = os.path.basename(filepath)
        m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
        if m:
            return datetime.datetime(*[int(g) for g in m.groups()])
        m = re.search(r'(\d{8})_(\d{6})', name)
        if m:
            d, t = m.group(1), m.group(2)
            return datetime.datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                                     int(t[:2]), int(t[2:4]), int(t[4:6]))
        return None

    def _norm(s):
        # Normaliza para comparação: remove acentos (NFKD → descarta marcas
        # combinantes → ASCII), minúsculas, mantém apenas letras/dígitos. Assim
        # 'POÇA' corresponde independentemente de como a cedilha está codificada
        # (NFC vs NFD), mantendo 'Ponto 1' distinto de 'Ponto 10'. A remoção de
        # acentos resolve nomes de pontos acentuados (POÇA, ÁGUA, SÃO, ...).
        s = unicodedata.normalize('NFKD', str(s))
        s = ''.join(ch for ch in s if not unicodedata.combining(ch))
        s = s.encode('ascii', 'ignore').decode('ascii')
        return re.sub(r'[^a-z0-9]', '', s.lower())

    def _file_site_match(filepath, site):
        # Subpastas por ponto: o ponto precisa ser um dos nomes de PASTA abaixo
        # de DRIVE_AUDIO_DIR (o nome do arquivo é ignorado aqui).
        rel     = os.path.relpath(filepath, DRIVE_AUDIO_DIR)
        folders = [_norm(p) for p in rel.split(os.sep)[:-1]]
        return _norm(site) in folders

    _multi_site = unique_recs['Point name'].nunique() > 1
    n_unscoped = 0   # pasta do ponto não encontrada
    n_notime   = 0   # nenhum arquivo no horário esperado

    for _, rec in unique_recs.iterrows():
        site = str(rec['Point name'])
        t0   = rec['_rec_dt']
        key  = (site, t0)

        # Restringe à subpasta deste ponto para que dois pontos gravados no
        # mesmo horário agendado nunca possam ser confundidos.
        site_files = [f for f in all_audio if _file_site_match(f, site)]

        if site_files:
            search = site_files
        elif _multi_site:
            # Falha de forma explícita em vez de adivinhar. Com vários pontos,
            # corresponder apenas pelo horário pode silenciosamente pegar a
            # gravação de outro ponto (os gravadores costumam estar sincronizados,
            # então todo ponto tem um arquivo neste horário exato). Um clipe
            # faltando é notado; um clipe errado não.
            print(f'  ERRO: nenhuma subpasta corresponde ao ponto "{site}" — ignorando esta '
                  f'gravação. Renomeie uma subpasta dentro do caminho de áudio para corresponder '
                  f'ao valor de "ponto" ("{site}") e execute novamente.')
            n_unscoped += 1
            continue
        else:
            # Ponto único: não há outro ponto com que confundir.
            search = all_audio

        time_hits = sorted(
            f for f in search
            if (fdt := _file_dt(f)) and abs((fdt - t0).total_seconds()) < 5
        )
        if not time_hits:
            print(f'  AVISO: nenhum arquivo de áudio para o ponto "{site}" em {t0} — ignorando.')
            n_notime += 1
            continue

        matched = time_hits[0]
        audio_file_map[key] = matched
        if len(time_hits) > 1:
            print(f'  AVISO: {len(time_hits)} arquivos correspondem ao ponto "{site}" em {t0}; '
                  f'usando {os.path.basename(matched)}.')
        else:
            print(f'  Correspondido : {site} -> {os.path.basename(matched)}')

    if n_unscoped:
        print(f'\n{n_unscoped} gravação(ões) ignorada(s) — pasta do ponto não encontrada. '
              f'Essas correspondências NÃO serão extraídas até que os nomes das pastas '
              f'correspondam à coluna "ponto".')
    if n_notime:
        print(f'{n_notime} gravação(ões) ignorada(s) — nenhum arquivo de áudio no horário esperado.')

print(f'\nArquivos de áudio prontos : {len(audio_file_map)} / {len(unique_recs)}')

In [61]:
#@title ✂️ Extrair e salvar segmentos { display-mode: "form" }


import os
import librosa, soundfile as sf
import numpy as np

saved   = 0
skipped = 0
errors  = 0

for _, row in df_ext.iterrows():
    if AUDIO_SOURCE == 'arbimon':
        key = (str(row['stream_id']), row['_rec_dt'])
    else:
        key = (str(row['Point name']), row['_rec_dt'])
    audio_path = audio_file_map.get(key)

    if not audio_path:
        errors += 1
        continue

    # start_time/end_time são segundos a partir do início da gravação (base 0).
    det_start  = float(row['start_time'])
    det_end    = float(row['end_time'])
    start_s    = max(0.0, det_start - SEGMENT_PADDING_S)
    end_s      = det_end + SEGMENT_PADDING_S
    conf       = float(row['Confidence'])

    site_safe  = str(row['Point name']).replace(' ', '_')
    label_safe = str(row['Common name']).replace(' ', '_').replace('/', '-')
    date_str   = str(row['Date']).replace('-', '')
    time_str   = str(row['Time']).replace(':', '')[:6]
    conf_str   = f'{conf:.3f}'
    # Mantém o início fracionário da detecção no nome para que duas correspondências no
    # mesmo segundo não sejam reduzidas ao mesmo nome de arquivo (e silenciosamente ignoradas).
    start_tag  = f'{det_start:.1f}'
    end_tag    = f'{det_end:.1f}'
    out_name   = f'{site_safe}_{date_str}_{time_str}_{start_tag}-{end_tag}s_{conf_str}_{label_safe}.wav'

    seg_out = os.path.join(DRIVE_SEGMENTS_DIR, site_safe, label_safe)
    os.makedirs(seg_out, exist_ok=True)
    out_path = os.path.join(seg_out, out_name)

    if os.path.exists(out_path):
        skipped += 1
        continue

    try:
        audio, _ = librosa.load(audio_path, sr=EXTRACT_SR, mono=True,
                                offset=start_s, duration=end_s - start_s)
    except Exception as e:
        print(f'  ERROR loading {os.path.basename(audio_path)}: {e}')
        errors += 1
        continue

    sf.write(out_path, audio, EXTRACT_SR)
    saved += 1

print(f'Segmentos salvos      : {saved}')
if skipped:
    print(f'Já existiam           : {skipped} (ignorados)')
if errors:
    print(f'Ignorados (sem áudio) : {errors}')
print(f'Pasta de saída        : {DRIVE_SEGMENTS_DIR}')

---
## Parte 6 — Revisar segmentos extraídos

Navegue pelos segmentos de áudio extraídos na Parte 5.
Cada segmento é mostrado como um espectrograma com seu rótulo e score de similaridade.
Você pode ouvi-lo com o player de áudio integrado e marcá-lo como **verdadeiro** (uma correspondência genuína) ou **falso** (um falso positivo).

- Marcar um segmento move seu arquivo para uma subpasta `true/` ou `false/` dentro da pasta de segmentos.
- Use **← Anterior** e **Próximo →** para navegar sem tomar uma decisão.
- Arquivos já revisados (dentro de `true/` ou `false/`) são automaticamente excluídos da lista a cada execução da célula.
- Os **controles do espectrograma** (tipo, faixa de frequência, faixa dinâmica em dB) atualizam a visualização instantaneamente sem recarregar o áudio.

> Execute a Parte 5 primeiro para que a pasta de segmentos seja preenchida, depois execute as duas células abaixo.

In [62]:
#@title ⚙️ Configurações de revisão { display-mode: "form" }
#@markdown Caminho completo para a pasta contendo os segmentos extraídos.
#@markdown Use o mesmo valor de **DRIVE_SEGMENTS_DIR** da Parte 5.
REVIEW_DIR = "/content/drive/MyDrive/vector_search_segments"  #@param {type:"string"}

#@markdown ---
#@markdown ### Padrões do espectrograma
#@markdown Estes definem os valores iniciais mostrados no revisor.
#@markdown Você pode ajustar todos eles ao vivo dentro da interface sem tem que executar esta célula novamente.

#@markdown **Tipo de espectrograma** — `mel` usa uma escala de frequência MEL; `fft` usa uma escala linear em Hz.
SPEC_TYPE = "mel"  #@param ["mel", "fft"]

#@markdown **Frequência mínima (Hz)** — limite inferior do eixo de frequência.
FREQ_MIN_HZ = 0  #@param {type:"integer"}

#@markdown **Frequência máxima (Hz)** — limite superior. Defina como **0** para usar a frequência de Nyquist (metade da taxa de amostragem).
FREQ_MAX_HZ = 0  #@param {type:"integer"}

#@markdown **Valor mínimo de dB** — menor valor mostrado na escala de cores (ex: −80). Faixa típica: −100 a −40.
DB_MIN = -80  #@param {type:"integer"}

import os
os.makedirs(os.path.join(REVIEW_DIR, 'true'),  exist_ok=True)
os.makedirs(os.path.join(REVIEW_DIR, 'false'), exist_ok=True)

import glob as _g
_all  = _g.glob(f'{REVIEW_DIR}/**/*.wav', recursive=True)
_skip = {os.path.join(REVIEW_DIR, 'true'), os.path.join(REVIEW_DIR, 'false')}
_pending = [f for f in _all
            if not any(os.path.commonpath([f, s]) == s for s in _skip)]

print(f'Pasta de segmentos : {REVIEW_DIR}')
print(f'Pendentes de revisão : {len(_pending)} segmento(s)')
print(f'Já verdadeiros     : {len(_g.glob(os.path.join(REVIEW_DIR, "true",  "*.wav")))}')
print(f'Já falsos          : {len(_g.glob(os.path.join(REVIEW_DIR, "false", "*.wav")))}')
print(f'Tipo de espectrograma : {SPEC_TYPE}')
print(f'Faixa de frequência : {FREQ_MIN_HZ} – {"Nyquist" if FREQ_MAX_HZ == 0 else str(FREQ_MAX_HZ) + " Hz"}')
print(f'Valor mínimo em dB : {DB_MIN} dB')

In [63]:
#@title 🎧 Revisor de segmentos { display-mode: "form" }


import os, glob, shutil, re, io, base64
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# PNG transparente 1x1 — atribuir b'' não limpa de forma confiável um widget
# Image no Colab, o que pode deixar o espectrograma do segmento anterior na
# tela ao lado do novo áudio. Use isto como um 'em branco' explícito.
_BLANK_PNG = base64.b64decode(
    'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg=='
)

# ── Coletar segmentos não revisados ───────────────────────────────────────────────
def _collect(base):
    skip = {os.path.join(base, 'true'), os.path.join(base, 'false')}
    return sorted(
        f for f in glob.glob(f'{base}/**/*.wav', recursive=True)
        if not any(os.path.commonpath([f, s]) == s for s in skip)
    )

# ── Extrair rótulo + escore do nome do arquivo ─────────────────────────────────────────
def _parse(filepath):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    # A confiança é o último token _<número>_ antes do rótulo; o prefixo guloso
    # força a correspondência sobre ele mesmo que haja outros decimais antes.
    m = re.search(r'.*_(\d+\.\d+)_(.+)$', stem)
    if m:
        return m.group(2).replace('_', ' '), float(m.group(1))
    return stem, None

# ── Estado ─────────────────────────────────────────────────────────────────────
_state = {'idx': 0, 'segs': _collect(REVIEW_DIR)}

# ── Controles do espectrograma ──────────────────────────────────────────────────────
_W_ctrl = {'style': {'description_width': 'initial'}}

_dd_type = widgets.Dropdown(
    options=[('Mel', 'mel'), ('FFT (linear)', 'fft')],
    value=globals().get('SPEC_TYPE', 'mel'),
    description='Tipo:',
    layout=widgets.Layout(width='165px'),
    **_W_ctrl,
)
_int_fmin = widgets.BoundedIntText(
    value=globals().get('FREQ_MIN_HZ', 0), min=0, max=96000, step=100,
    description='Hz mín:',
    layout=widgets.Layout(width='155px'),
    **_W_ctrl,
)
_int_fmax = widgets.BoundedIntText(
    value=globals().get('FREQ_MAX_HZ', 0), min=0, max=96000, step=100,
    description='Hz máx:',
    layout=widgets.Layout(width='155px'),
    **_W_ctrl,
)
_int_db = widgets.BoundedIntText(
    value=globals().get('DB_MIN', -80), min=-120, max=-20, step=5,
    description='Piso dB:',
    layout=widgets.Layout(width='160px'),
    **_W_ctrl,
)
_ctrl_row = widgets.HBox(
    [_dd_type, _int_fmin, _int_fmax, _int_db,
     widgets.Label('(Hz máx = 0 → Nyquist)', layout=widgets.Layout(margin='6px 0 0 4px'))],
    layout=widgets.Layout(justify_content='center', margin='4px 0', flex_wrap='wrap'),
)

# ── Widgets de exibição ───────────────────────────────────────────────────────────
_spec_widget  = widgets.Image(value=b'', format='png',
                               layout=widgets.Layout(width='720px', height='auto'))
_audio_widget = widgets.HTML(value='')
_lbl_prog     = widgets.HTML(value='')
_lbl_error    = widgets.HTML(value='')

# ── Linha de navegação ────────────────────────────────────────────────────────────
_BW = widgets.Layout(width='120px', height='36px')
_btn_prev  = widgets.Button(description='← Anterior', layout=widgets.Layout(width='110px', height='36px'))
_btn_true  = widgets.Button(description='✔  Verdadeiro', button_style='success', layout=_BW)
_btn_false = widgets.Button(description='✘  Falso', button_style='danger',  layout=_BW)
_btn_next  = widgets.Button(description='Próximo →', layout=widgets.Layout(width='110px', height='36px'))

_nav_row = widgets.HBox(
    [_btn_prev, _btn_true, _btn_false, _btn_next],
    layout=widgets.Layout(justify_content='center', margin='8px 0'),
)

# ── Linha de entrada do rótulo falso (oculta até Falso ser pressionado) ─────────────────────
_input_label       = widgets.Text(
    placeholder='Rótulo correto  (deixe em branco → "unknown")',
    layout=widgets.Layout(width='340px', height='36px'),
)
_btn_confirm_false = widgets.Button(description='Confirmar', button_style='warning',
                                     layout=widgets.Layout(width='100px', height='36px'))
_btn_cancel        = widgets.Button(description='Cancelar',
                                     layout=widgets.Layout(width='80px',  height='36px'))

_label_row = widgets.VBox([
    widgets.HTML('<span style="font-size:12px">Insira o rótulo correto para este segmento:</span>'),
    widgets.HBox(
        [_input_label, _btn_confirm_false, _btn_cancel],
        layout=widgets.Layout(justify_content='center', margin='4px 0'),
    ),
], layout=widgets.Layout(align_items='center', margin='4px 0', display='none'))

# ── UI completa ───────────────────────────────────────────────────────────────────
_ui = widgets.VBox([
    _lbl_prog,
    _ctrl_row,
    _spec_widget,
    _audio_widget,
    _lbl_error,
    _label_row,
    _nav_row,
], layout=widgets.Layout(align_items='center'))

# ── Renderização do espectrograma ─────────────────────────────────────────────────────
def _show_spec():
    segs = _state['segs']
    if not segs:
        _spec_widget.value = _BLANK_PNG
        return
    filepath = segs[_state['idx']]
    label, conf = _parse(filepath)
    conf_str  = f'{conf:.3f}' if conf is not None else '?'
    spec_type = _dd_type.value
    fmin_hz   = _int_fmin.value
    db_floor  = _int_db.value
    _lbl_error.value = ''
    try:
        y, sr   = librosa.load(filepath, sr=None, mono=True)
        fmax_hz = _int_fmax.value if _int_fmax.value > 0 else sr // 2
        fig, ax = plt.subplots(figsize=(9, 4))
        if spec_type == 'mel':
            S   = librosa.feature.melspectrogram(
                y=y, sr=sr, n_mels=128, fmin=fmin_hz, fmax=fmax_hz)
            Sd  = librosa.power_to_db(S, ref=np.max)
            img = librosa.display.specshow(
                Sd, sr=sr, x_axis='time', y_axis='mel', ax=ax,
                fmin=fmin_hz, fmax=fmax_hz, vmin=db_floor, vmax=0)
        else:
            D   = librosa.stft(y)
            Sd  = librosa.amplitude_to_db(np.abs(D), ref=np.max)
            img = librosa.display.specshow(
                Sd, sr=sr, x_axis='time', y_axis='hz', ax=ax,
                vmin=db_floor, vmax=0)
            ax.set_ylim(fmin_hz, fmax_hz)
        ax.set_title(f'{label}   |   escore: {conf_str}', fontsize=13, fontweight='bold')
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        plt.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        _spec_widget.value = buf.read()
    except Exception as e:
        plt.close('all')
        _spec_widget.value = _BLANK_PNG
        _lbl_error.value = f'<span style="color:red">Erro no espectrograma: {e}</span>'

def _show_audio():
    segs = _state['segs']
    if not segs:
        _audio_widget.value = ''
        return
    try:
        with open(segs[_state['idx']], 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        _audio_widget.value = (
            f'<audio controls style="width:720px;margin-top:4px">'
            f'<source src="data:audio/wav;base64,{b64}" type="audio/wav">'
            f'</audio>'
        )
    except Exception as e:
        _audio_widget.value = f'<span style="color:red">Erro de áudio: {e}</span>'

def _show():
    segs = _state['segs']
    n    = len(segs)
    if n == 0:
        _lbl_prog.value     = '<b>Todos os segmentos foram revisados.</b>'
        _spec_widget.value  = _BLANK_PNG
        _audio_widget.value = ''
        return
    _lbl_prog.value = (
        f'<span style="font-size:13px">Segmento <b>{_state["idx"] + 1}</b> de <b>{n}</b></span>'
    )
    _show_spec()
    _show_audio()

# ── Operações de arquivo ───────────────────────────────────────────────────────────
def _move(verdict, subfolder=None):
    segs = _state['segs']
    if not segs:
        return
    dest_dir = os.path.join(REVIEW_DIR, verdict, subfolder) if subfolder \
               else os.path.join(REVIEW_DIR, verdict)
    os.makedirs(dest_dir, exist_ok=True)
    src = segs.pop(_state['idx'])
    shutil.move(src, os.path.join(dest_dir, os.path.basename(src)))
    if _state['idx'] >= len(segs) and _state['idx'] > 0:
        _state['idx'] -= 1
    _show()

def _nav(delta):
    n = len(_state['segs'])
    if n == 0:
        return
    _state['idx'] = max(0, min(n - 1, _state['idx'] + delta))
    _show()

# ── Handlers dos botões ───────────────────────────────────────────────────────────
def _show_label_input(_):
    _input_label.value        = ''
    _label_row.layout.display = ''
    _nav_row.layout.display   = 'none'

def _on_confirm_false(_):
    raw      = _input_label.value.strip()
    subfolder = raw.replace(' ', '_').replace('/', '-') if raw else 'unknown'
    _label_row.layout.display = 'none'
    _nav_row.layout.display   = ''
    _move('false', subfolder)

def _on_cancel(_):
    _label_row.layout.display = 'none'
    _nav_row.layout.display   = ''

_btn_true.on_click(lambda _: _move('true'))
_btn_false.on_click(_show_label_input)
_btn_confirm_false.on_click(_on_confirm_false)
_btn_cancel.on_click(_on_cancel)
_btn_prev.on_click(lambda _: _nav(-1))
_btn_next.on_click(lambda _: _nav(+1))

# ── Observadores de mudança do espectrograma ─────────────────────────────────────────────────────
def _on_spec_change(change):
    if _state['segs']:
        _show_spec()

for _ctrl in (_dd_type, _int_fmin, _int_fmax, _int_db):
    _ctrl.observe(_on_spec_change, names='value')

# ── Iniciar ────────────────────────────────────────────────────────────────────
_show()
display(_ui)